# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**My lane: Lane 2, Refresh / Content Opportunity Scoring.** The question from W01 and W02 stays the same: which pages should a reviewer look at first? In this notebook I write the data contract for that question on the real warehouse release, prove three facts about my slice with three queries, build five features and then spring the leakage trap on purpose.

**How to run it:** in Colab, save your Hugging Face READ token as a Secret named `HF_TOKEN` (key panel on the left) and run all cells. Outside Colab the notebook reads the `HF_TOKEN` environment variable, or asks for it with `getpass`. The token is never printed and never written into the notebook.

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import getpass

import duckdb
import numpy as np
import pandas as pd

# The repo is public, so the token comes from a Colab Secret or an environment variable.
# It is never printed. getpass is only the last fallback.
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face READ token: ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
print("DuckDB connected. Hugging Face token loaded (not printed).")

DuckDB connected. Hugging Face token loaded (not printed).


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Contract answers 1 to 3

**1. What one row means.**
In the source table `fact_content_daily_performance` one row is one page on one day: `report_date` × `client_hash_id` × `content_hash_id`. For my lane I collapse March into one row per page. So in my feature frame **one row = one page (`content_hash_id`) with its search numbers for March 2026, as seen at the end of 31 March 2026.**

**2. Tables I use.**
- `fact_content_daily_performance`, partition `month=2026-03`: all three checks and all five features.
- `fact_content_daily_performance`, partition `month=2026-04`: only to build the label. No feature reads April.
- `dim_clients`: only `gsc_data_start`, to keep pages of clients whose GSC tracking had started by 1 March. For a client that started later, an early March zero means "not tracked yet", not "no impressions".

**3. Time window.**

| Window | Dates | Used for |
|---|---|---|
| Feature window | 1 to 31 March 2026 (31 days) | the five features |
| Decision moment | end of 31 March 2026 | everything a feature uses must be known by then |
| Label window | 1 to 30 April 2026 (30 days) | the label `declined_next_month` |
| Sealed month | June 2026 (the `_sample` table is this month) | never loaded here, kept as the final test month |

March sits in the middle of the panel (daily data runs from 2025-01-27 to 2026-06-30), so even the label month is two months away from the sealed one. One honest note: in real life GSC data arrives 2 to 3 days late, so a live version of this would run around 3 April. In this frozen snapshot March is already complete.

In [3]:
REL = "hf://datasets/FlyRank/internship-warehouse"

# Contract windows and rules. Every cell below uses these, so the contract and the code cannot drift apart.
FEATURE_MONTH = "2026-03"          # feature window: 1 to 31 March 2026
LABEL_MONTH = "2026-04"            # label window: 1 to 30 April 2026
DECISION_MOMENT = "2026-03-31"     # end of this day
MARCH_DAYS, APRIL_DAYS = 31, 30
LATE_MARCH_START = "2026-03-16"    # start of the second half of March (one momentum feature)
GSC_TRACKED_BY = "2026-03-01"      # client GSC tracking must cover the whole feature window
MIN_MARCH_IMPRESSIONS = 100        # below this a 20% drop is mostly noise
DECLINE_CUTOFF_PCT = -20           # same 20% rule as the starter "down" label

con.execute(f"CREATE OR REPLACE VIEW march AS SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month={FEATURE_MONTH}/*.parquet')")
con.execute(f"CREATE OR REPLACE VIEW april AS SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month={LABEL_MONTH}/*.parquet')")
con.execute(f"CREATE OR REPLACE VIEW dim_clients AS SELECT * FROM read_parquet('{REL}/dim_clients.parquet')")

print(f"Feature window : month={FEATURE_MONTH} ({MARCH_DAYS} days)")
print(f"Decision moment: end of {DECISION_MOMENT}")
print(f"Label window   : month={LABEL_MONTH} ({APRIL_DAYS} days)")
print("June 2026 and the _sample table: not loaded anywhere in this notebook")

Feature window : month=2026-03 (31 days)
Decision moment: end of 2026-03-31
Label window   : month=2026-04 (30 days)
June 2026 and the _sample table: not loaded anywhere in this notebook


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Contract answers 4 and 5

**4. What I predict or rank (label or proxy).**
I rank pages by how likely they are to lose search visibility next month. The label is `declined_next_month = 1` when a page's average daily GSC impressions in April 2026 are more than 20% below its average in March 2026. I compare per day because March has 31 days and April has 30. The 20% rule is the same one the starter data uses for `trend_direction == "down"`. This is a **proxy**: a decline says the page is worth a look. It does not prove that a refresh will bring the traffic back.

**5. One thing I deliberately exclude.**
The page edit dates in `dim_content` (`content_updated_date` and `last_optimized_date`). "Days since last update" is the most natural refresh feature, but `dim_content` is a snapshot from the July 2026 export. For a page that was edited in April, May or June it shows that later date, not the date I would have seen on 31 March. Using it would quietly give the model information from after the decision moment, so I leave the whole table out of this notebook.

### Every field I touch, in one bucket

| Bucket | Fields | Why |
|---|---|---|
| Feature inputs (March only) | `gsc_impressions`, `gsc_clicks`, `gsc_sum_position`, `gsc_avg_position`, `gsc_data_available` | observed before the decision moment |
| Label (April only) | April `gsc_impressions` → `declined_next_month` | the outcome I rank for |
| Label-derived, never a feature | `imp_change_pct` (April vs March change per day) | the label is a cutoff on it. Used once in the trap below, then deleted |
| Context | `content_hash_id`, `client_hash_id`, `report_date`, `dim_clients.gsc_data_start` | page key, grouped split, window filters. Never learned from |
| Excluded | `dim_content` edit dates | snapshot from July 2026, after my decision moment (answer 5) |
| Excluded | `fact_content_query_90d` | its fixed 90 day window is the most recent ~3 months of the snapshot, so it overlaps my April label |
| Excluded | GA4 columns (`ga4_*`, `sessions_*`, `ai_*`, `scroll_events`) | only a small share of March rows have GA4 data (query 3). The rest are zero-filled or NULL |

In [4]:
FEATURES = ["imp_per_day_log", "ctr_pct", "avg_position", "late_march_share", "gsc_days"]
LABEL = "declined_next_month"
LEAK_COLUMN = "imp_change_pct"     # label-derived: only used in the trap, then deleted
CONTEXT = ["content_hash_id", "client_hash_id"]

assert len(FEATURES) <= 5 and LABEL not in FEATURES and LEAK_COLUMN not in FEATURES
print("Features       :", FEATURES)
print("Label          :", LABEL)
print("Context only   :", CONTEXT)
print("Never a feature:", LEAK_COLUMN)

Features       : ['imp_per_day_log', 'ctr_pct', 'avg_position', 'late_march_share', 'gsc_days']
Label          : declined_next_month
Context only   : ['content_hash_id', 'client_hash_id']
Never a feature: imp_change_pct


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Three verification queries on `month=2026-03`

Each query checks one claim from the contract. They read the March partition straight from Hugging Face. DuckDB only pulls the columns a query needs, so nothing big is downloaded.

**Query 1: grain.** Claim: one row = one page on one day, so `report_date` + `client_hash_id` + `content_hash_id` never repeats. Also every page belongs to exactly one client, so `content_hash_id` alone can be the key of my page level frame.

In [5]:
q1 = con.sql("""
    WITH per_key AS (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
        FROM march
        GROUP BY report_date, client_hash_id, content_hash_id
    )
    SELECT
        SUM(n)::BIGINT                                    AS rows_in_march,
        COUNT(*)                                          AS unique_day_page_keys,
        COUNT(*) FILTER (WHERE n > 1)                     AS keys_seen_twice_or_more,
        COUNT(DISTINCT content_hash_id)                   AS pages,
        COUNT(DISTINCT (client_hash_id, content_hash_id)) AS client_page_pairs
    FROM per_key
""").df()
q1

,rows_in_march,unique_day_page_keys,keys_seen_twice_or_more,pages,client_page_pairs
0,9841378,9841378,0,331437,331437


Result: 9,841,378 rows and 9,841,378 unique day-page keys, with 0 keys seen twice. There are 331,437 pages and 331,437 client-page pairs, so no page is shared by two clients. **The grain holds.** `content_hash_id` is a safe key for my one row per page frame.

**Query 2: row count and date span.** Claim: my slice is the whole March partition, one row per page per day, from 1 to 31 March 2026.

In [6]:
q2 = con.sql("""
    SELECT
        COUNT(*)                        AS rows_in_slice,
        MIN(report_date)                AS first_day,
        MAX(report_date)                AS last_day,
        COUNT(DISTINCT report_date)     AS days_covered,
        COUNT(DISTINCT client_hash_id)  AS clients,
        COUNT(DISTINCT content_hash_id) AS pages
    FROM march
""").df()
q2

,rows_in_slice,first_day,last_day,days_covered,clients,pages
0,9841378,2026-03-01,2026-03-31,31,55,331437


Result: 9,841,378 rows from 2026-03-01 to 2026-03-31 with all 31 days present, from 55 clients and 331,437 pages. That is about 29.7 rows per page, so most pages have a row on most days, even on days with no impressions (query 3 shows those rows are zero-filled). Not every page has all 31 days, which is why the feature build below checks it.

**Query 3: availability.** Claim: only some March rows carry real GSC data. I keep a row only when `gsc_data_available IS TRUE`. I write `IS TRUE` on purpose: these flags can be NULL (the GA4 flag is NULL for clients without GA4). `IS TRUE` never returns NULL, so "kept" and "not kept" always add up to all rows. If I split rows with `= TRUE` and `= FALSE` (or `flag` and `NOT flag`), a NULL row lands on neither side and gets lost without any error.

In [7]:
q3 = con.sql("""
    SELECT
        COUNT(*)                                                                     AS rows_in_slice,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)                           AS gsc_rows_kept,
        ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) / COUNT(*), 1) AS gsc_kept_pct,
        COUNT(DISTINCT content_hash_id) FILTER (WHERE gsc_data_available IS TRUE)    AS pages_with_gsc_data,
        SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS NOT TRUE)::BIGINT  AS impressions_on_dropped_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)                           AS ga4_rows_kept,
        ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) AS ga4_kept_pct,
        COUNT(*) FILTER (WHERE ga4_data_available IS NULL)                           AS ga4_flag_null
    FROM march
""").df()
q3.astype(object).T.rename(columns={0: "value"})   # shown as one tall column so it is easy to read

,value
rows_in_slice,9841378
gsc_rows_kept,3611061
gsc_kept_pct,36.7
pages_with_gsc_data,176738
impressions_on_dropped_rows,0
ga4_rows_kept,413966
ga4_kept_pct,4.2
ga4_flag_null,3018741


Result: 3,611,061 of 9,841,378 rows survive `gsc_data_available IS TRUE` (36.7%). They cover 176,738 of the 331,437 pages. The dropped rows hold 0 impressions in total, so they are zero-filled days with no search data, not lost traffic. This means sums of impressions and clicks are the same with or without the filter. Position is where the filter matters, so my position feature only uses `IS TRUE` days with a position above 0 (Google never shows position 0).

GA4 is much thinner: only 4.2% of March rows have `ga4_data_available IS TRUE` and 3,018,741 rows have a NULL flag. This is why all five of my features are GSC only.

### Five features (built from the same March month)

This is a build step, not a fourth check. One query sums March per page (plus April impressions, only for the label). Then pandas applies four eligibility rules and shows how many pages survive each one:

1. the page has a row on all 31 days of March. A page that starts mid-month has a short history: its missing days are absent, not zero
2. its client's GSC tracking started on or before 1 March 2026 (`dim_clients.gsc_data_start`). Clients with no GSC start date drop out here too
3. at least 100 GSC impressions in March, because below that a 20% drop is mostly noise
4. a row on all 30 days of April, so the label can be measured

In [8]:
pages = con.sql(f"""
    WITH march_pages AS (
        SELECT
            content_hash_id,
            ANY_VALUE(client_hash_id)                           AS client_hash_id,  -- safe: query 1 showed one client per page
            COUNT(*)                                            AS march_rows,
            SUM(gsc_impressions)::BIGINT                        AS imp_march,
            SUM(gsc_clicks)::BIGINT                             AS clicks_march,
            SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '{LATE_MARCH_START}')::BIGINT AS imp_late_march,
            COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)  AS gsc_days,
            SUM(gsc_sum_position) FILTER (WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0)::BIGINT AS position_sum,
            SUM(gsc_impressions)  FILTER (WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0)::BIGINT AS position_impressions
        FROM march
        GROUP BY content_hash_id
    ),
    april_pages AS (  -- label window only: no feature is built from this
        SELECT content_hash_id, COUNT(*) AS april_rows, SUM(gsc_impressions)::BIGINT AS imp_april
        FROM april
        GROUP BY content_hash_id
    )
    SELECT m.*, c.gsc_data_start, a.april_rows, a.imp_april
    FROM march_pages m
    LEFT JOIN dim_clients c USING (client_hash_id)
    LEFT JOIN april_pages a USING (content_hash_id)
""").df()

gsc_start = pd.to_datetime(pages["gsc_data_start"])
rules = [
    ("pages with rows in March", pd.Series(True, index=pages.index)),
    ("row on all 31 days of March", pages["march_rows"] == MARCH_DAYS),
    ("client GSC tracking started by 1 March", gsc_start <= pd.Timestamp(GSC_TRACKED_BY)),
    (f"at least {MIN_MARCH_IMPRESSIONS} GSC impressions in March", pages["imp_march"] >= MIN_MARCH_IMPRESSIONS),
    ("row on all 30 days of April (label measurable)", pages["april_rows"] == APRIL_DAYS),
]
keep = pd.Series(True, index=pages.index)
funnel = []
for rule_name, rule in rules:
    keep &= rule
    funnel.append((rule_name, int(keep.sum())))
eligible = pages[keep].reset_index(drop=True)
pd.DataFrame(funnel, columns=["rule (applied in order)", "pages left"])

,rule (applied in order),pages left
0,pages with rows in March,331437
1,row on all 31 days of March,275133
2,client GSC tracking started by 1 March,267986
3,at least 100 GSC impressions in March,94496
4,row on all 30 days of April (label measurable),94496


In [9]:
# The five features: March numbers only
frame = pd.DataFrame({
    "content_hash_id": eligible["content_hash_id"],
    "client_hash_id": eligible["client_hash_id"],
    "imp_per_day_log": np.log1p(eligible["imp_march"] / MARCH_DAYS),
    "ctr_pct": 100 * eligible["clicks_march"] / eligible["imp_march"],
    "avg_position": eligible["position_sum"] / eligible["position_impressions"],
    "late_march_share": eligible["imp_late_march"] / eligible["imp_march"],
    "gsc_days": eligible["gsc_days"],
})
frame[FEATURES] = frame[FEATURES].astype("float64")

# The label: April (label window) compared with March, per day
march_per_day = eligible["imp_march"] / MARCH_DAYS
april_per_day = eligible["imp_april"] / APRIL_DAYS
april_change_pct = (100 * (april_per_day - march_per_day) / march_per_day).astype("float64")   # label-derived, NOT a feature
frame[LABEL] = (april_change_pct < DECLINE_CUTOFF_PCT).astype(int)

assert frame["content_hash_id"].is_unique, "grain broken: expected one row per page"
assert frame[FEATURES].notna().all().all(), "a feature has missing values"
print(f"Feature frame: {len(frame):,} pages x {len(FEATURES)} features, from {frame['client_hash_id'].nunique()} clients")
print(f"Base rate of {LABEL}: {frame[LABEL].mean():.3f}")
print(f"Pages with avg_position below 1: {(frame['avg_position'] < 1).sum():,} ({(frame['avg_position'] < 1).mean():.1%})")
frame[FEATURES + [LABEL]].head()

Feature frame: 94,496 pages x 5 features, from 38 clients
Base rate of declined_next_month: 0.516
Pages with avg_position below 1: 462 (0.5%)


,imp_per_day_log,ctr_pct,avg_position,late_march_share,gsc_days,declined_next_month
0,3.016483,0.664452,4.365449,0.669435,29.0,0
1,3.300604,0.123457,4.624691,0.423457,29.0,1
2,4.109816,0.322928,1.920344,0.585038,30.0,0
3,2.906372,0.186567,4.509328,0.565299,29.0,1
4,2.833213,0.604839,3.766129,0.536290,29.0,1


In [10]:
frame[FEATURES].describe().round(3)

,imp_per_day_log,ctr_pct,avg_position,late_march_share,gsc_days
count,94496.000,94496.000,94496.000,94496.000,94496.000
mean,3.489,0.248,14.597,0.534,29.430
std,1.368,0.396,14.983,0.165,3.486
min,1.441,0.000,0.020,0.000,1.000
25%,2.341,0.000,4.896,0.441,29.000
50%,3.321,0.118,8.383,0.529,31.000
75%,4.471,0.350,19.209,0.617,31.000
max,9.899,15.584,115.662,1.000,31.000


**Every feature, with its "available when?" line**

| # | Feature | How it is built (March only) | Knowable at the decision moment because… |
|---|---|---|---|
| 1 | `imp_per_day_log` | log(1 + March impressions / 31) | it only adds up impressions from 1 to 31 March, which are all in before the decision moment |
| 2 | `ctr_pct` | 100 × March clicks / March impressions | March clicks and March impressions are both complete by the end of 31 March |
| 3 | `avg_position` | impression-weighted average position over `gsc_data_available IS TRUE` days with a position above 0 | it only uses March days that had GSC data. No April number goes into it |
| 4 | `late_march_share` | impressions on 16 to 31 March / all March impressions (about 0.52 means flat) | both halves of March end on or before 31 March |
| 5 | `gsc_days` | number of March days with `gsc_data_available IS TRUE` (0 to 31) | each day's flag is set when that day's GSC data arrives. All 31 days are inside March |

The frame has 94,496 pages from 38 clients. The label base rate is 0.516, so about half of these pages lost more than 20% of their daily impressions from March to April.

One thing I noticed: 462 pages (0.5%) have an average position below 1, which Google never shows. I do not know the cause yet. It is too small to change this week's result, so I keep these pages and will look at it again in the signal audit.

### The trap: one label-derived column, on purpose

**Quick score** = mean ROC AUC of a depth-4 decision tree over 5 folds. Each fold tests on clients the tree never saw in training (`GroupKFold` on `client_hash_id`, the same held-out-client idea as in W02). 0.5 means a coin flip and 1.0 means perfect.

- Step 1: honest score with my five features.
- Step 2: add `imp_change_pct`, the April vs March change per day. My label is simply `imp_change_pct < -20`, so this column is the answer in disguise, exactly like `trend_pct` in notebook 02.
- Step 3: delete it and score again.

In [11]:
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier, export_text


def quick_score(columns):
    """ROC AUC per fold. 5 folds grouped by client, so test clients are never seen in training."""
    tree = DecisionTreeClassifier(max_depth=4, random_state=42)
    return cross_val_score(tree, frame[columns], frame[LABEL], groups=frame["client_hash_id"],
                           cv=GroupKFold(n_splits=5), scoring="roc_auc")


honest = quick_score(FEATURES)
print("Step 1, honest (5 features):     ROC AUC per fold", honest.round(3), f"-> mean {honest.mean():.3f}")

Step 1, honest (5 features):     ROC AUC per fold [0.674 0.691 0.72  0.694 0.68 ] -> mean 0.692


In [12]:
frame[LEAK_COLUMN] = april_change_pct          # added ON PURPOSE: it is built from the label window
leaky = quick_score(FEATURES + [LEAK_COLUMN])
print("Step 2, + imp_change_pct (leak): ROC AUC per fold", leaky.round(3), f"-> mean {leaky.mean():.3f}")

# What the tree learned: its very first split is the label rule itself
leaky_tree = DecisionTreeClassifier(max_depth=1, random_state=42).fit(frame[FEATURES + [LEAK_COLUMN]], frame[LABEL])
print(export_text(leaky_tree, feature_names=FEATURES + [LEAK_COLUMN]))

Step 2, + imp_change_pct (leak): ROC AUC per fold [1. 1. 1. 1. 1.] -> mean 1.000
|--- imp_change_pct <= -20.00
|   |--- class: 1
|--- imp_change_pct >  -20.00
|   |--- class: 0



In [13]:
frame = frame.drop(columns=[LEAK_COLUMN])      # leak deleted
assert LEAK_COLUMN not in frame.columns

honest_again = quick_score(FEATURES)
print("Step 3, leak deleted (5 features): ROC AUC per fold", honest_again.round(3), f"-> mean {honest_again.mean():.3f}")

pd.DataFrame({
    "step": ["1. honest, 5 features", "2. + imp_change_pct (leak)", "3. leak deleted, 5 features"],
    "mean ROC AUC": [honest.mean(), leaky.mean(), honest_again.mean()],
    "lowest fold": [honest.min(), leaky.min(), honest_again.min()],
    "highest fold": [honest.max(), leaky.max(), honest_again.max()],
}).round(3)

Step 3, leak deleted (5 features): ROC AUC per fold [0.674 0.691 0.72  0.694 0.68 ] -> mean 0.692


,step,mean ROC AUC,lowest fold,highest fold
0,"1. honest, 5 features",0.692,0.674,0.72
1,2. + imp_change_pct (leak),1.000,1.000,1.00
2,"3. leak deleted, 5 features",0.692,0.674,0.72


**What happened.** With my five honest features the quick score is **0.692** (folds from 0.674 to 0.720). When I added `imp_change_pct` it jumped to **1.000 on every fold**. The depth-1 tree shows why: its only split is `imp_change_pct <= -20`, which is my label rule copied back. Nothing was learned here. The model just read the answer from the label window. After deleting the column the score is back to 0.692.

**The honest number I keep is 0.692.** It is modest, but it is real: it only uses what was known on 31 March and it is measured on clients the tree never saw.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation: a few clients dominate my slice and the label moves with the client, not only with the page.**

My frame has 38 clients, but the top 3 hold 52.0% of all pages. Among the 16 clients with 1,000+ pages, the April decline rate goes from 27.1% to 94.4% (code below). So part of what `declined_next_month` measures is "this client had a bad April", for example a site wide change or a seasonal dip in its market. No single page refresh can fix that and my five page features cannot see it. It also makes the quick score depend on which clients land in the test fold (0.674 to 0.720 above). With only one March to April pair I cannot separate a client wide dip from real page decay.

What I will do about it later: report scores per client and try a label that compares each page with its own client's change.

In [14]:
by_client = (frame.groupby("client_hash_id")[LABEL]
             .agg(pages="size", decline_rate="mean")
             .sort_values("pages", ascending=False)
             .reset_index(drop=True))            # client ids dropped: rank only
by_client.index = by_client.index + 1
by_client.index.name = "client rank (by pages)"

top3_share = by_client["pages"].head(3).sum() / by_client["pages"].sum()
big = by_client[by_client["pages"] >= 1000]
print(f"Clients in the frame: {len(by_client)}. The top 3 hold {top3_share:.1%} of all pages.")
print(f"Decline rate across the {len(big)} clients with 1,000+ pages: {big['decline_rate'].min():.1%} to {big['decline_rate'].max():.1%}")
print(f"Honest quick score across folds: {honest_again.min():.3f} to {honest_again.max():.3f}")
by_client.head(10).round(3)

Clients in the frame: 38. The top 3 hold 52.0% of all pages.
Decline rate across the 16 clients with 1,000+ pages: 27.1% to 94.4%
Honest quick score across folds: 0.674 to 0.720


,pages,decline_rate
client rank (by pages),,
1,21238,0.412
2,16337,0.729
3,11539,0.489
4,8501,0.295
5,7498,0.271
6,7368,0.718
7,3406,0.427
8,2532,0.524
9,2418,0.417


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Checked against "what done looks like":**

- [x] Five plain-words contract answers: answers 1 to 3 in section 1, answers 4 and 5 in section 2
- [x] Exactly three verification queries with their outputs visible: grain, row count + date span, availability with `IS TRUE`
- [x] A five-feature frame with an "available when?" line for every feature
- [x] The deliberate leak shown (score jumps to 1.000) and removed (honest number kept)
- [x] One named limitation of my slice (section 4)